In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [62]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()


In [ ]:
model = Pipeline([
    ("scalar", StandardScaler()), 
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)) # z = w^T * x + b. gradient based descent
])

model.fit(X_train, y_train)

In [ ]:
logistic_regression = model["model"] # extract trained model
scalar = model["scalar"] # get scalar used by model

X_train_scaled = pd.DataFrame(scalar.transform(X_train), columns=X_train.columns) # transform train and test data
X_test_scaled = pd.DataFrame(scalar.transform(X_test), columns=X_test.columns)


masker = shap.maskers.Independent(X_train_scaled) # background dataset to simulate missing features, assumes features are independent and replaces with values sampled from X_train
explainer = shap.LinearExplainer(logistic_regression, masker=masker) # compute SHAP values

shap_values = explainer.shap_values(X_train_scaled) # compute contribution from each feature in classification
# phi_i(row) = weight_i * (x_i(row) - mean_i), for shap values with linear model

print(f"SHAP values shape: {shap_values.shape}") # match X_train shape?
print(X_train_scaled)

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, show=False) #-ve = retained, +ve = churn
plt.title("Feature impact on churn predicted value by SHAP value")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_train_scaled, plot_type="bar", show=False)
plt.title("Ranking of feature importance by mean SHAP value")
plt.tight_layout()
plt.show()

In [ ]:
feature_churn_importance = pd.DataFrame({
    "feature": X_train.columns,
    "mean_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_shap", ascending=False) # by descending

print("Top 10 features that drive churn:\n")
print(feature_churn_importance.head(10).to_string(index=False))

In [ ]:
highest_three_features_by_churn_importance = feature_churn_importance["feature"].head(3).tolist()

for feature in highest_three_features_by_churn_importance:
    shap.dependence_plot(feature, shap_values, X_train_scaled, show=False)
    plt.title(f"SHAP Dependence — {feature}")
    plt.tight_layout()
    plt.show()

# This is to check if churn risk increases linearly with feature values or if there is a threshold effect

In [ ]:
churned_index = y_test[y_test == 1].index[0] # get index of first churned customer 

# Compute SHAP explanations for the whole set of test data
shap_values_test = explainer(X_test_scaled)

# return a single-row Explanation object if X_test_scaled has same index as y_test
single_explanation = shap_values_test[churned_index]

# Plot waterfall for specific customer
shap.plots.waterfall(single_explanation, show=False)

plt.title("Biggest features that caused specific customer to churn")
plt.tight_layout()
plt.show()

In [ ]:
churned_index = y_test[y_test == 0].index[0] # get index of first retained customer

# Compute SHAP explanations for the whole set of test data
shap_values_test = explainer(X_test_scaled)

# return a single-row Explanation object if X_test_scaled has same index as y_test
single_explanation = shap_values_test[churned_index]

# Plot waterfall for specific customer
shap.plots.waterfall(single_explanation, show=False)

plt.title("Biggest features that caused specific customer to stay")
plt.tight_layout()
plt.show()